In [1]:
import pandas as pd
import numpy as np


## Checklist Before Building a Model

| Step | Check |
|---|---|
| Missing values | `df.isnull().sum()` -- none remaining |
| Categorical columns | All converted to numbers |
| Feature scales | Standardized or normalized |
| Target distribution | Skewness checked, transform applied if needed |
| Outliers | Investigated and handled |
| Split integrity | Scalers and imputers fit on train only |
| Data types | All columns are `float32` before passing to PyTorch |

In [2]:
df = pd.read_csv("../data/AmesHousing.csv")


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non


# Missing Values
Most models cannot handle NaN values. Passing a tensor with NaN will silently propagate through the network, producing NaN loss from the first step.

```python
df.isnull().sum()                     # count per column
df.isnull().mean() * 100              # percentage per column
df.isnull().sum().sum()               # total missing across dataset
```

# types

In [4]:
print(df.isnull().sum()) # will show every columns, filtering out 0s makes more sense

null_counts = df.isnull().sum()
null_counts[null_counts > 0]

Order               0
PID                 0
MS SubClass         0
MS Zoning           0
Lot Frontage      490
                 ... 
Mo Sold             0
Yr Sold             0
Sale Type           0
Sale Condition      0
SalePrice           0
Length: 82, dtype: int64


Lot Frontage       490
Alley             2732
Mas Vnr Type      1775
Mas Vnr Area        23
Bsmt Qual           80
Bsmt Cond           80
Bsmt Exposure       83
BsmtFin Type 1      80
BsmtFin SF 1         1
BsmtFin Type 2      81
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Electrical           1
Bsmt Full Bath       2
Bsmt Half Bath       2
Fireplace Qu      1422
Garage Type        157
Garage Yr Blt      159
Garage Finish      159
Garage Cars          1
Garage Area          1
Garage Qual        159
Garage Cond        159
Pool QC           2917
Fence             2358
Misc Feature      2824
dtype: int64

In [5]:
df_isnull_percentage = df.isnull().mean() * 100

df_isnull_percentage[null_counts > 0]

Lot Frontage      16.723549
Alley             93.242321
Mas Vnr Type      60.580205
Mas Vnr Area       0.784983
Bsmt Qual          2.730375
Bsmt Cond          2.730375
Bsmt Exposure      2.832765
BsmtFin Type 1     2.730375
BsmtFin SF 1       0.034130
BsmtFin Type 2     2.764505
BsmtFin SF 2       0.034130
Bsmt Unf SF        0.034130
Total Bsmt SF      0.034130
Electrical         0.034130
Bsmt Full Bath     0.068259
Bsmt Half Bath     0.068259
Fireplace Qu      48.532423
Garage Type        5.358362
Garage Yr Blt      5.426621
Garage Finish      5.426621
Garage Cars        0.034130
Garage Area        0.034130
Garage Qual        5.426621
Garage Cond        5.426621
Pool QC           99.556314
Fence             80.477816
Misc Feature      96.382253
dtype: float64

## Exploring Null Values

exploring values where lot frontage is null

In [6]:
df[df["Lot Frontage"].isnull()]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
11,12,527165230,20,RL,NaN,7980,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,Shed,500,3,2010,WD,Normal,185000
14,15,527182190,120,RL,NaN,6820,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,6,2010,WD,Normal,212000
22,23,527368020,60,FV,NaN,7500,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,1,2010,WD,Normal,216000
23,24,527402200,20,RL,NaN,11241,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Shed,700,3,2010,WD,Normal,149000
24,25,527402250,20,RL,NaN,12537,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,149900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2894,2895,916326010,20,RL,NaN,16669,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,1,2006,WD,Normal,228000
2897,2898,916403130,60,RL,NaN,11170,Pave,NaN,IR2,Lvl,...,0,NaN,GdPrv,NaN,0,4,2006,WD,Normal,250000
2898,2899,916460070,20,RL,NaN,8098,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,10,2006,WD,Normal,202000
2912,2913,923226150,90,RL,NaN,11836,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,3,2006,WD,Normal,146500


# Solutions for Nan Values (based on amount of nan values)

| Reason | Example | Solution |
|---|---|---|
| Structurally absent | No pool, so pool quality is NaN | Fill with a sentinel value (`"None"`, `0`) |
| Randomly not recorded | Measurement skipped | Impute with median (numerical) or mode (categorical) |
| Missing in bulk (>50%) | Column was rarely collected | Consider dropping the column entirely |


# Types/Reasons of Missing Values


* MCAR (Missing Completely At Random): the missingness has no relationship to any variable, observed or not. It's essentially noise. Rare in real-world data.
    * A lab's blood sample tubes broke during shipping due to a random equipment malfunction, causing some blood_pressure readings to be lost. This has nothing to do with the patient's age, health, or any other variable in the dataset, it's pure chance.

* MAR (Missing at Random): the missingness can be explained by other observed variables in the dataset.
    * blood_pressure is missing mostly for elderly patients, because a different protocol skipped that measurement for older patients. The missingness depends on age (an observed variable), not on the blood pressure value itself.

* MNAR (Missing Not At Random): the missingness is caused by the missing value itself (e.g. people with high incomes choosing not to report income). No imputation method fully fixes this, because the reason for the missingness isn't captured anywhere in your data.

# Imputation


For numerical/**contionus** values:
you can use mean or median values as the strategy

For categorical values, text based ones, where mean or median doesnt make sense,
mode could be used.

also setting a constant value is possible.

### **MissingIndicator** helps creating a binary tag to later figure out which columns are tagged.

```
from sklearn.impute import SimpleImputer

# Numerical
num_imputer = SimpleImputer(strategy="median")

# Categorical
cat_imputer = SimpleImputer(strategy="most_frequent")

# Constant
const_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")

```

# Other Advanced Imputers:

## Not Implemented in Scikit Learn, Group Based Imputation
Instead of using the global median, compute the median within groups defined by another column. This is often more accurate because the true typical value differs across subgroups.


## KNN Imputer

Instead of using a single global statistic, it finds the K nearest neighbors
of a row (based on other features) and uses their values to fill in the missing one.

Uses inter-row relationships rather than column-wide statistics.

When to use:
- When the data has structure/correlation between features
- When missing values are not completely random (MAR assumption)
- When the dataset is not too large (distance computation is expensive)

Watch out:
- **Scaling is mandatory: KNN is distance-based, so features must be on the same scale
  otherwise high-magnitude features will dominate the distance calculation**
- Has an n_neighbors hyperparameter, default is 5

from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=5)

## Iterative Imputer

Models each feature with missing values as a function of the other features,
then iterates until convergence.

Each column takes a turn as the target variable. The other columns are used as
features, a regression model is fit, and missing values are predicted.
This repeats until the imputations stabilize.

Still experimental in scikit-learn, requires an explicit enable import.

When to use:
- When there are strong relationships between features
- When you want more sophisticated imputation than mean/median or KNN
- When missing values have a structural cause

Watch out:
- Default estimator is BayesianRidge, but other regressors can be swapped in
  (e.g. RandomForestRegressor)
- max_iter controls how many rounds of imputation are performed
- Can be slower than KNN on large datasets

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iter_imputer = IterativeImputer(max_iter=10, random_state=42)


# Simple Imputation

Lot Frontage = linear distance of propertys boundary that directly borders a street

might be correlated with / effected by Lot Area, Lot Shape and Lot Config

In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

#### 2 functions of sckiti transformers, fit and transform and fit_transform:

In [8]:

imputer.fit(df[['Lot Frontage']]) # learning the median
df['Lot Frontage'] = imputer.transform(df[['Lot Frontage']])

# or together

df['Lot Frontage'] = imputer.fit_transform(df[['Lot Frontage']])


### Group-based imputation

Instead of using the global median, compute the median within groups defined by another column. This is often more accurate because the true typical value differs across subgroups.

Chatgpt: categorial columns are better than continious ones like 1St Flr Lf or Lot Area, as


In [9]:
filled = (
    df.groupby("Neighborhood")["Lot Frontage"] #groups lot frontage with neighboorhood
    .transform(lambda x: x.fillna(x.median()))  #fills in each neighborhood group the empty values with its median
)
filled

0       141.0
1        80.0
2        81.0
3        93.0
4        74.0
        ...  
2925     37.0
2926     68.0
2927     62.0
2928     77.0
2929     74.0
Name: Lot Frontage, Length: 2930, dtype: float64

In [10]:
filled.isnull().sum()

np.int64(0)

# Writing Own Transformer / Imputer

# Estimator:
- responsilibity: learning from the value.
- fit(X,y) returns self

# fit():
It's the "learning" phase. You show the estimator your training data, it extracts whatever statistics or patterns it needs, and stores them on self. After fit() runs, the object is stateful. It knows something about the data it saw.

## Why it returns self
So you can chain: self.fit(X).transform(X). Also a general scikit-learn convention for all methods that modify state.

### BaseEstimator
- get_params()

Returns the parameters passed to __init__ as a dictionary.
For example if you wrote __init__(self, strategy="median", fill_value=0), then get_params() returns {"strategy": "median", "fill_value": 0}.

BaseEstimator can do this automatically because it uses Python's inspect module to look at your __init__ signature. This is why you must set every parameter with the exact same name in __init__: self.strategy = strategy. Otherwise it can't find them.

set_params()
The reverse of get_params(). Lets you set parameters from outside the object.
pythonimputer.set_params(strategy="mean")

#### Who actually calls these

GridSearchCV: calls set_params() on each iteration to try different hyperparameter combinations, and get_params() to compare results.
Pipeline: when chaining multiple steps together, it uses double-underscore notation to access each step's parameters. For example:
```
pipeline.set_params(imputer__strategy="mean")
```
Here imputer is the name of the step in the pipeline, and strategy is your __init__ parameter. This double-underscore notation works because of set_params() under the hood.

# Transformer:
A special type of estimator, it learns + transforms the data.

## Transform()

* It's the "applying" phase. It takes data, uses what was learned in fit(), and returns a modified version of that data. It does not learn anything new. It only applies.

* It must not modify self. **No new attributes, no updating self.medians_.** The learned state is frozen after fit(). If transform() were allowed to update state, calling it on test data would corrupt what was learned from training data.

* It returns transformed data, same shape as input.
    * For an imputer, same number of rows and columns, just with NaNs filled in.

### TransformerMixin

TransformerMixin is a very small class. It only adds one thing: fit_transform().
Its implementation is essentially:
```
 fit_transform(self, X, y=None):
    return self.fit(X, y).transform(X)
```

# Implementing GroupImputer 

In [14]:
from sklearn.base import TransformerMixin
from sklearn.base import BaseEstimator
from sklearn.utils.validation import check_is_fitted



class GroupImputer(BaseEstimator, TransformerMixin):

    def __init__(self, group_col, value_col, impute_method="median"):
        self.group_col = group_col
        self.value_col = value_col
        self.impute_method = impute_method

    def fit(self, X, y=None):
        # Statistics object look like:
        # A    50.0
        # B    80.0
        # C    60.0

        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[self.group_col, self.value_col])

        if self.impute_method == "median":
            self.statistics_ = X.groupby(self.group_col)[self.value_col].median()
        elif self.impute_method == "mean":
            self.statistics_ = X.groupby(self.group_col)[self.value_col].mean()


        self.global_statistic_ = X[self.value_col].median()


        return self

    def transform(self, X):
        check_is_fitted(self)

        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[self.group_col, self.value_col])

        X = X.copy()


        def fill(row):
            if pd.isna(row[self.value_col]):
                # gets group_cols' value saved in statistics object, or if its empty global statistic
                return self.statistics_.get(row[self.group_col], self.global_statistic_)
            return row[self.value_col]

        filled = X.apply(fill, axis=1)

        # filled.to_numpy() returns 1D array: shape (3,)
        # [50. 80. 60.]
        #
        # .reshape(-1, 1) converts to 2D array: shape (3, 1)
        # [[50.]
        #  [80.]
        #  [60.]]
        #
        # ColumnTransformer expects 2D, so reshape is required

        return filled.to_numpy().reshape(-1, 1)









In [12]:
import pytest
import ipytest
ipytest.autoconfig()
#-v :verbose, detailed output


In [17]:
%%ipytest

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "Neighborhood": ["A", "A", "B", "B"],
        "Lot Frontage": [50.0, np.nan, 80.0, np.nan],
    })

def test_fill(sample_df):
    imputer = GroupImputer(group_col="Neighborhood", value_col="Lot Frontage", impute_method="median")

    imputer_output = imputer.fit_transform(sample_df).squeeze()

    pandas_output = sample_df.groupby("Neighborhood")["Lot Frontage"] .transform(lambda x: x.fillna(x.median()))



    np.testing.assert_array_equal(imputer_output, pandas_output)

def test_unseen_neighborhood(sample_df):

    imputer = GroupImputer(group_col="Neighborhood", value_col="Lot Frontage", impute_method="median")
    imputer.fit(sample_df)

    unseen_df = pd.DataFrame({ # D wasnt on test set
        "Neighborhood": ["D"],
        "Lot Frontage": [np.nan],
    })

    result = imputer.transform(unseen_df)
    assert result[0,0] == imputer.global_statistic_


..                                                                                           [100%]
2 passed in 0.01s


# Knn-Imputer